In [ ]:
import matplotlib.pyplot
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds

In [ ]:
physical_devices = tf.config.list_physical_devices("GPU")
tf.config.experimental.set_memory_growth(physical_devices[0], True)

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "mnist",
    split=["train", "test"],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

In [ ]:
print(ds_info)

In [ ]:
def normalize_img(image, label):
    return tf.cast(image, tf.float32)/255.0, label

In [ ]:
AUTOTUNE = tf.data.experimental.AUTOTUNE
BATCH_SIZE = 64

In [ ]:
ds_train = ds_train.map(normalize_img, num_parallel_calls=AUTOTUNE)
ds_train = ds_train.cache()
ds_train = ds_train.shuffle(1000)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(AUTOTUNE)

In [ ]:
ds_test = ds_test.map(normalize_img, num_parallel_calls=AUTOTUNE)
ds_test = ds_test.batch(128)
ds_test = ds_test.prefetch(AUTOTUNE)

In [ ]:
import keras
from keras import layers

model = keras.Sequential([
    keras.Input((28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),  # Changed pool size
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),  # Changed pool size
    layers.Conv2D(256, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),  # Changed pool size
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),  # Changed pool size
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),  # Added a dense layer
    layers.Dropout(0.5),  # Added dropout layer
    layers.Dense(10, activation='softmax')  # Softmax activation for classification
])


In [ ]:
model.summary()

In [ ]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [ ]:
model.fit(ds_train, epochs=30, verbose=1)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model.save_weights('saved_model.weights.h5')

In [ ]:
model.save('complete_saved_model.keras')

In [ ]:
from huggingface_hub import login

login()


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id="ThorBaller/mnist_CNN_model", repo_type="model")


In [ ]:
from huggingface_hub import HfApi

# Initialize the API
api = HfApi()

# Save your model
api.upload_file(
    path_or_fileobj="/kaggle/working/complete_saved_model.keras",
    path_in_repo="complete_saved_model.keras",
    repo_id="ThorBaller/mnist_1_CNN_model",
    repo_type="model"
)
